## CT4101 - Milestone 2
# 1 dataset loading and exploration
This project will explore whether patient demographics, behaviours and history can predict a positive cervical cancer result using the UCI Cervical Cancer (Risk Factors) dataset

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load the dataset
df = pd.read_csv(
    "../data/risk_factors_cervical_cancer.csv",
    na_values=["?"]
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (858, 36)


,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),IUD,...,STDs: Time since first diagnosis,STDs: Time since last diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology,Biopsy
0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0,0,0,0,0,0,0,0
1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0,0,0,0,0,0,0,0
2,34,1.0,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0,0,0,0,0,0,0,0
3,52,5.0,16.0,4.0,1.0,37.0,37.0,1.0,3.0,0.0,...,NaN,NaN,1,0,1,0,0,0,0,0
4,46,3.0,21.0,4.0,0.0,0.0,0.0,1.0,15.0,0.0,...,NaN,NaN,0,0,0,0,0,0,0,0


## 2 - data audit 
# 2.1 data types and any missing values
I am going to examine the dataset for missing values and variables before determining which is the best pre-processing method for this 

In [11]:
# check data types
print("Data types:")
print(df.dtypes.value_counts())

# count missing values per column
missing_values = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})

# display columns containing missing values
display(
    missing_values[missing_values["Missing Count"] > 0]
    .sort_values("Missing Percentage", ascending=False)
)

Data types:
float64    26
int64      10
Name: count, dtype: int64


,Missing Count,Missing Percentage
STDs: Time since first diagnosis,787,91.72
STDs: Time since last diagnosis,787,91.72
IUD (years),117,13.64
IUD,117,13.64
Hormonal Contraceptives,108,12.59
Hormonal Contraceptives (years),108,12.59
STDs (number),105,12.24
STDs,105,12.24
STDs:vulvo-perineal condylomatosis,105,12.24
STDs:vaginal condylomatosis,105,12.24


# 2.2 duplicates and target distribution
I'm going to check for duplicate rows and see how many patients have positive or negative biopsy results


In [12]:
# check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# check number of positive/negative biopsy results
print("\nBiopsy result counts:")
print(df["Biopsy"].value_counts())

# percentage of each biopsy result
print("\nBiopsy result percentages:")
print((df["Biopsy"].value_counts(normalize=True) * 100).round(2))

# double-check for missing values in the target variable
print("\nMissing values in Biopsy column:")
print("Columns with missing values:", df.isna().any().sum())



Duplicate rows: 23

Biopsy result counts:
Biopsy
0    803
1     55
Name: count, dtype: int64

Biopsy result percentages:
Biopsy
0    93.59
1     6.41
Name: proportion, dtype: float64

Missing values in Biopsy column:
Columns with missing values: 26


Output: 
There are 23 duplicate rows in the dataset. This is something I need to consider before splitting the data as this could effect the evaluation results. As stated before the biopsy results are quite imbalanced as there are 803 negative biopsy results and there are 55 positive biopsy results. This is important to note as a model could achieve high accuracy by predicting negative for each patient without identifing any positive cases. Which is why I need to put emphasis on recall when doing this assignment. There are also 26 columns with missing values which I will also need to handle during pre processing.

# 2.3 check for outliers
I'm gonna take a look at the summary statistics to see if there are any unusual values which may need to be looked into

In [13]:
display(df.describe().T)

# examine numerical features closer
columns_to_examine = [
    "Age", "Number of sexual partners", "First sexual intercourse", "Num of pregnancies", "Smokes (years)"]

display(df[columns_to_examine].describe().T)

,count,mean,std,min,25%,50%,75%,max
Age,858.0,26.820513,8.497948,13.0,20.0,25.0,32.0,84.0
Number of sexual partners,832.0,2.527644,1.667760,1.0,2.0,2.0,3.0,28.0
First sexual intercourse,851.0,16.995300,2.803355,10.0,15.0,17.0,18.0,32.0
Num of pregnancies,802.0,2.275561,1.447414,0.0,1.0,2.0,3.0,11.0
Smokes,845.0,0.145562,0.352876,0.0,0.0,0.0,0.0,1.0
Smokes (years),845.0,1.219721,4.089017,0.0,0.0,0.0,0.0,37.0
Smokes (packs/year),845.0,0.453144,2.226610,0.0,0.0,0.0,0.0,37.0
Hormonal Contraceptives,750.0,0.641333,0.479929,0.0,0.0,1.0,1.0,1.0
Hormonal Contraceptives (years),750.0,2.256419,3.764254,0.0,0.0,0.5,3.0,30.0
IUD,741.0,0.112011,0.315593,0.0,0.0,0.0,0.0,1.0


,count,mean,std,min,25%,50%,75%,max
Age,858.0,26.820513,8.497948,13.0,20.0,25.0,32.0,84.0
Number of sexual partners,832.0,2.527644,1.667760,1.0,2.0,2.0,3.0,28.0
First sexual intercourse,851.0,16.995300,2.803355,10.0,15.0,17.0,18.0,32.0
Num of pregnancies,802.0,2.275561,1.447414,0.0,1.0,2.0,3.0,11.0
Smokes (years),845.0,1.219721,4.089017,0.0,0.0,0.0,0.0,37.0


output findings:
There are a few high values in the dataset. For example the maximum number of sexual partners is 28 compared to the medium of 2, and the maximum number of pregnencies is 11 compared to a median of 2. I'm not going to remove them now however I will consider their impact during preprocessing and modellng 

In [14]:
# noting which columns have more then half of values missing
missing_values = df.isna().mean() * 100

print("\nColumns with more than 50% missing values:")
print(missing_values[missing_values > 50])

# check columns with one one unique non missing value
print("\nColumns with only one unique non-missing value:")
print(df.columns[df.nunique(dropna=True) == 1].tolist())


Columns with more than 50% missing values:
STDs: Time since first diagnosis    91.724942
STDs: Time since last diagnosis     91.724942
dtype: float64

Columns with only one unique non-missing value:
['STDs:cervical condylomatosis', 'STDs:AIDS']


Final thoughts of my data audit:
The dataset contains 858 records and 36 columns. There are 23 duplicate rows however these may represent different pateints with the same information which I haven't removed yet. The target value is imbalanced which will be important when evaluating the baseline model. There are 26 columns with missing values. Two columns are both related to the timing of the stf diagnosis so I will probably exclude them. I also found two std related columns with only one unique recorded value which I will exclude as they dont provide any useful information. 

# 3. data preprocessing
I'm going to remove columns now with too much data missing, columns with no variation and test results which dont align with my prediction question

In [15]:
# columns to exclude
columns_to_exclude = [
    "STDs: Time since first diagnosis",
    "STDs: Time since last diagnosis",
    "STDs:cervical condylomatosis",
    "STDs:AIDS",
    "Hinselmann",
    "Schiller",
    "Citology"
]

# divide input features and target variable
X = df.drop(columns=columns_to_exclude + ["Biopsy"])
Y = df["Biopsy"]

print("\nInput features shape:", X.shape)
print("Target variable shape:", Y.shape)
print("Missing values in target variable:", Y.isna().sum())


Input features shape: (858, 28)
Target variable shape: (858,)
Missing values in target variable: 0


In [18]:
# train/test split is based on the approach outlined in week 4 of the lectures
# stratification is used as the biopsy results are imbalanced

from sklearn.model_selection import train_test_split

# split data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

print("\nTraining set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\n positive biopsey percentage in training set:")
print(round(Y_train.mean() * 100, 2))

print("\n positive biopsey percentage in testing set:")
print(round(Y_test.mean() * 100, 2))


Training set shape: (686, 28)
Testing set shape: (172, 28)

 positive biopsey percentage in training set:
6.41

 positive biopsey percentage in testing set:
6.4


# 3.1 preprocessing pipeline
I'm going to use median imputation for numerical features and most frequent imputation for binary features. Im also going to standardise the numerical features as some have larger ranges then others. When I'm using KNN later on this will be needed. Preprocessing will be fit only with training data to avoid data leakage


In [ ]:
# preprocessing approach based on week 3 lecture
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# identify binary and continuous features
binary_cols = [
    col for col in X_train.columns
    if set(X_train[col].dropna().unique()).issubset({0, 1})
]

numeric_cols = [
    col for col in X_train.columns
    if col not in binary_cols
]


